# Session 08: Camera Models & Calibration

**CVI4IC - Summer Semester 2026**

Understanding the pinhole camera model, lens distortion, and calibration.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

## 1. The Camera Matrix

In [ ]:
# Example intrinsic camera matrix
fx, fy = 800, 800       # focal length in pixels
cx, cy = 320, 240       # principal point

K = np.array([[fx,  0, cx],
              [ 0, fy, cy],
              [ 0,  0,  1]], dtype=np.float64)

print("Camera Matrix K:")
print(K)

# Project a 3D point
point_3d = np.array([0.5, 0.3, 2.0])  # X, Y, Z in camera frame
point_2d_h = K @ point_3d
point_2d = point_2d_h[:2] / point_2d_h[2]

print(f"\n3D Point: {point_3d}")
print(f"2D Projection: ({point_2d[0]:.1f}, {point_2d[1]:.1f})")

## 2. Simulating Lens Distortion

In [ ]:
# Create a grid image
grid = np.ones((480, 640, 3), dtype=np.uint8) * 255
for x in range(0, 640, 40):
    cv2.line(grid, (x, 0), (x, 480), (0, 0, 0), 1)
for y in range(0, 480, 40):
    cv2.line(grid, (0, y), (640, y), (0, 0, 0), 1)

# Simulate barrel distortion
dist_coeffs = np.array([-0.3, 0.1, 0, 0, 0], dtype=np.float64)

# Generate undistortion maps
map1, map2 = cv2.initUndistortRectifyMap(K, dist_coeffs, None, K, (640, 480), cv2.CV_32FC1)
# Apply "distortion" by reverse mapping
distorted = cv2.remap(grid, map1, map2, cv2.INTER_LINEAR)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(grid)
axes[0].set_title("Undistorted Grid")
axes[1].imshow(distorted)
axes[1].set_title("Distorted Grid")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. Chessboard Detection (Calibration Prep)

In [ ]:
# Create a synthetic chessboard image
board = np.zeros((400, 500), dtype=np.uint8)
square_size = 40
offset_x, offset_y = 50, 60

for row in range(8):
    for col in range(10):
        if (row + col) % 2 == 0:
            x = offset_x + col * square_size
            y = offset_y + row * square_size
            cv2.rectangle(board, (x, y), (x + square_size, y + square_size), 255, -1)

# Try to find chessboard corners
ret, corners = cv2.findChessboardCorners(board, (9, 7), None)

board_color = cv2.cvtColor(board, cv2.COLOR_GRAY2BGR)
if ret:
    cv2.drawChessboardCorners(board_color, (9, 7), corners, ret)
    print(f"Found {len(corners)} corners")
else:
    print("Chessboard corners not found (try adjusting pattern size)")

plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(board_color, cv2.COLOR_BGR2RGB))
plt.title("Chessboard Corner Detection")
plt.axis("off")
plt.show()

## 4. Perspective Transform (Homography)

In [ ]:
# Create a document-like image viewed at an angle
doc = np.ones((300, 400, 3), dtype=np.uint8) * 240
cv2.putText(doc, "Hello CV!", (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 0), 3)

# Define source and destination points
src_pts = np.float32([[0, 0], [400, 0], [400, 300], [0, 300]])
dst_pts = np.float32([[50, 30], [350, 50], [380, 280], [20, 260]])

# Compute homography and warp
H = cv2.getPerspectiveTransform(src_pts, dst_pts)
warped = cv2.warpPerspective(doc, H, (400, 300))

# Unwarp
H_inv = cv2.getPerspectiveTransform(dst_pts, src_pts)
unwarped = cv2.warpPerspective(warped, H_inv, (400, 300))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cv2.cvtColor(doc, cv2.COLOR_BGR2RGB))
axes[0].set_title("Original")
axes[1].imshow(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB))
axes[1].set_title("Warped (perspective)")
axes[2].imshow(cv2.cvtColor(unwarped, cv2.COLOR_BGR2RGB))
axes[2].set_title("Unwarped")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Exercises

1. Modify the camera matrix parameters and observe how projection changes
2. Calibrate a camera using multiple chessboard images (if you have them)
3. Use `cv2.findHomography` with RANSAC to align two images with matched features

In [ ]:
# Your code here
